<a href="https://colab.research.google.com/github/srihanreddy/Reinforcement-Learning/blob/main/Lab_Assignment_10_Bellman_Optimality_for_Deterministic_Environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Task 1 & 2: Inspect MDP and Verify Transitions
We define the 2x2 Grid World environment with states $S_0$, $S_1$, $G$ (Goal), and $W$ (Wall/Obstacle). We define the states, actions, transition function, and rewards.

In [1]:
import numpy as np
import pandas as pd

# Define States and Actions
states = ['S0', 'S1', 'G']  # G is terminal. S0, S1 are non-terminal.
actions = ['Up', 'Down', 'Left', 'Right']
gamma = 0.9
theta = 1e-6

# Transition Mapping and Rewards: (state, action) -> (next_state, reward)
# Let's assume a 2x2 layout:
# [ S0 ] [ S1 ]
# [ W  ] [ G  ]
# Where W is a wall. S0 is top-left, S1 is top-right, G is bottom-right (Goal).
# Going into a Wall or out of bounds returns the agent to the same state with reward -1.
transitions = {
    ('S0', 'Up'): ('S0', -1),       # Out of bounds
    ('S0', 'Down'): ('S0', -1),     # Hitting the wall (W)
    ('S0', 'Left'): ('S0', -1),     # Out of bounds
    ('S0', 'Right'): ('S1', -1),    # Move to S1

    ('S1', 'Up'): ('S1', -1),       # Out of bounds
    ('S1', 'Down'): ('G', 10),      # Reach Goal!
    ('S1', 'Left'): ('S0', -1),     # Move to S0
    ('S1', 'Right'): ('S1', -1)     # Out of bounds
}

# Print MDP Info
print("=== MDP Information ===")
print(f"States: {states}")
print(f"Actions: {actions}")
print(f"Discount Factor (gamma): {gamma}\n")

# Verify Transitions
print("=== Transition Verification ===")
transition_records = []
for (s, a), (s_next, r) in transitions.items():
    print(f"State: {s:2s} | Action: {a:5s} -> Next State: {s_next:2s} | Reward: {r:3d}")
    transition_records.append({'Current State': s, 'Action': a, 'Next State': s_next, 'Reward': r})

# Check if all non-terminal state-action pairs have transitions defined
all_valid = all((s, a) in transitions for s in ['S0', 'S1'] for a in actions)
print(f"\nAll state-action pairs defined: {all_valid}")

=== MDP Information ===
States: ['S0', 'S1', 'G']
Actions: ['Up', 'Down', 'Left', 'Right']
Discount Factor (gamma): 0.9

=== Transition Verification ===
State: S0 | Action: Up    -> Next State: S0 | Reward:  -1
State: S0 | Action: Down  -> Next State: S0 | Reward:  -1
State: S0 | Action: Left  -> Next State: S0 | Reward:  -1
State: S0 | Action: Right -> Next State: S1 | Reward:  -1
State: S1 | Action: Up    -> Next State: S1 | Reward:  -1
State: S1 | Action: Down  -> Next State: G  | Reward:  10
State: S1 | Action: Left  -> Next State: S0 | Reward:  -1
State: S1 | Action: Right -> Next State: S1 | Reward:  -1

All state-action pairs defined: True


### Task 3, 4 & 5: Value Iteration with Synchronous Updates

In [2]:
# Task 3: Initialize state values
V = {s: 0.0 for s in states} # V(G) is initialized to 0 and will remain 0
print("Initial State Values:", V)

# Task 4 & 5: Value Iteration
iteration = 0
converged = False
history = []

print("\n=== Starting Value Iteration ===")
while not converged:
    V_new = V.copy() # Synchronous update holding copy
    delta = 0

    for s in ['S0', 'S1']: # Update only non-terminal states
        q_values = []
        for a in actions:
            s_next, r = transitions[(s, a)]
            q_val = r + gamma * V[s_next]
            q_values.append(q_val)

        V_new[s] = max(q_values)
        delta = max(delta, abs(V_new[s] - V[s]))

    V = V_new
    iteration += 1
    history.append({'Iteration': iteration, 'V(S0)': V['S0'], 'V(S1)': V['S1'], 'V(G)': V['G'], 'Delta': delta})
    print(f"Iteration {iteration:2d} | V(S0): {V['S0']:.4f} | V(S1): {V['S1']:.4f} | Delta: {delta:.6f}")

    if delta < theta:
        converged = True

print(f"\nValue Iteration converged in {iteration} iterations.")

Initial State Values: {'S0': 0.0, 'S1': 0.0, 'G': 0.0}

=== Starting Value Iteration ===
Iteration  1 | V(S0): -1.0000 | V(S1): 10.0000 | Delta: 10.000000
Iteration  2 | V(S0): 8.0000 | V(S1): 10.0000 | Delta: 9.000000
Iteration  3 | V(S0): 8.0000 | V(S1): 10.0000 | Delta: 0.000000

Value Iteration converged in 3 iterations.


### Task 6 & 7: Calculate Action Values (Q-values) and Extract Optimal Policy

In [3]:
Q = {}
optimal_policy = {}

print("=== Final Q-Values and Optimal Policy ===")
for s in ['S0', 'S1']:
    Q[s] = {}
    best_q = -float('inf')
    best_actions = []

    for a in actions:
        s_next, r = transitions[(s, a)]
        q_val = r + gamma * V[s_next]
        Q[s][a] = q_val
        print(f"Q*({s}, {a:5s}) = {q_val:8.4f}")

        if q_val > best_q + 1e-9: # handling floating point precision
            best_q = q_val
            best_actions = [a]
        elif abs(q_val - best_q) < 1e-9:
            best_actions.append(a)

    optimal_policy[s] = best_actions
    print(f"Optimal Policy for {s}: {best_actions} with value {best_q:.4f}\n")

=== Final Q-Values and Optimal Policy ===
Q*(S0, Up   ) =   6.2000
Q*(S0, Down ) =   6.2000
Q*(S0, Left ) =   6.2000
Q*(S0, Right) =   8.0000
Optimal Policy for S0: ['Right'] with value 8.0000

Q*(S1, Up   ) =   8.0000
Q*(S1, Down ) =  10.0000
Q*(S1, Left ) =   6.2000
Q*(S1, Right) =   8.0000
Optimal Policy for S1: ['Down'] with value 10.0000



### Task 8 & 9: Verify and Explain Results

In [4]:
# Verification: V*(s) == max_a Q*(s, a)
print("=== Numerical Verification ===")
for s in ['S0', 'S1']:
    max_q = max(Q[s].values())
    v_val = V[s]
    is_equal = np.isclose(v_val, max_q)
    print(f"State {s} | V*(s) = {v_val:.6f} | max_a Q*(s, a) = {max_q:.6f} | Matches? {is_equal}")

=== Numerical Verification ===
State S0 | V*(s) = 8.000000 | max_a Q*(s, a) = 8.000000 | Matches? True
State S1 | V*(s) = 10.000000 | max_a Q*(s, a) = 10.000000 | Matches? True


### Explanation of Results

1. **Optimal Actions**:
   - For state $S_1$, the optimal action is **Down** because it directly leads to the Goal state $G$ with a large immediate reward of $+10$, giving the highest action value.
   - For state $S_0$, the optimal action is **Right**, which transitions the agent to state $S_1$ so that it can quickly reach the goal state $G$ on the subsequent step.

2. **Influence of Goal Reward**:
   - The large positive reward $+10$ at $G$ propagates backward to $S_1$ and subsequently to $S_0$. Since reaching $G$ is highly lucrative, values of states closer to $G$ are significantly higher.

3. **Importance of Discount Factor ($\gamma = 0.9$)**:
   - The discount factor ensures that rewards obtained in the future are worth less than immediate rewards, modeling a preference for shorter paths to the goal and guaranteeing mathematical convergence of the Bellman updates.